initial set-up

In [9]:
''' caio '''

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from datetime import timedelta

df = pd.read_csv('Dati_progetto_completo.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%m%d%Y %H:%M')
df = df.sort_values('timestamp').reset_index(drop=True)

df['date'] = df['timestamp'].dt.normalize()
df['hour'] = df['timestamp'].dt.hour + 1     # 1..24
df['dow']  = df['timestamp'].dt.dayofweek  
print(df)

       ZONEID           timestamp  Forecasted Total Load  \
0           1 2011-01-01 00:00:00                  15187   
1           1 2011-01-01 01:00:00                  14464   
2           1 2011-01-01 02:00:00                  13940   
3           1 2011-01-01 03:00:00                  13609   
4           1 2011-01-01 04:00:00                  13391   
...       ...                 ...                    ...   
25963       1 2013-12-17 19:00:00                  23091   
25964       1 2013-12-17 20:00:00                  22504   
25965       1 2013-12-17 21:00:00                  21538   
25966       1 2013-12-17 22:00:00                  20025   
25967       1 2013-12-17 23:00:00                  18306   

       Forecasted Zonal Load  Zonal Price       date  hour  dow  
0                       5091        43.17 2011-01-01     1    5  
1                       4918        36.24 2011-01-01     2    5  
2                       4763        34.64 2011-01-01     3    5  
3              

matrix day x hour

In [2]:
prices = df.pivot_table(index = 'date', columns = 'hour', values = 'Zonal Price', ).sort_index()
loads = df.pivot_table(index = 'date', columns = 'hour', values = 'Forecasted Zonal Load')


logaritmichal tranformation (staibilized the variance)

In [3]:
log_p = np.log(prices)
log_load = np.log(loads)

remove the Nan values of the price column

In [4]:
valid = log_p.dropna().index
log_p, log_load = log_p.loc[valid], log_load.loc[valid]

ARX function
 pd,h = βh,0 + βh,1pd−1,h + βh,2pd−2,h + βh,3pd−7,h + βh,4pmin + βh,5zt + i∈{1,6,7} βh,i+5Di + εd,h

In [ ]:
def build_training(target_date, h, window_start):
    X, y = [], []
    d = window_start + timedelta(days=7)
    while d < target_date:
        try: 
            X.append([
                log_p.loc[d - timedelta(days = 1), h], 
                log_p.loc[d - timedelta(days=2), h],
                log_p.loc[d - timedelta(days=7), h],
                log_p.loc[d - timedelta(days=1)].min(),
                log_load.loc[d, h],
                *[1.0 if d.dayofweek == k else 0.0 for k in range(6)]
            ])
            y.append(log_p.loc[d,h])
        except KeyError:
            pass
        d += timedelta(days = 1)
    return np.array(X), np.array(y)

target_date = pd.Timestamp('2012-12-29')
h = 15 
T = 728


(array([[4.15731936, 4.050915  , 3.71673778, ..., 0.        , 0.        ,
        1.        ],
       [3.9076133 , 4.15731936, 3.67351198, ..., 0.        , 0.        ,
        0.        ],
       [3.87660306, 3.9076133 , 3.93398017, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [3.51927667, 3.5975862 , 3.66458676, ..., 0.        , 0.        ,
        0.        ],
       [3.73074091, 3.51927667, 3.6493587 , ..., 1.        , 0.        ,
        0.        ],
       [3.73361379, 3.73074091, 3.6044103 , ..., 0.        , 1.        ,
        0.        ]], shape=(721, 11)), array([3.9076133 , 3.87660306, 4.19086613, 4.14820124, 4.48277654,
       4.48582343, 4.70147987, 4.12713439, 3.91202301, 4.13196143,
       4.35632414, 4.14170513, 4.24190213, 4.36348072, 4.75359019,
       4.94648781, 5.13573961, 4.82542958, 4.22800095, 4.24290824,
       4.66729956, 4.37033354, 4.18935167, 4.50722654, 4.47471976,
       4.25291404, 4.19539581, 4.22814675, 3.92572865, 3.96271612,
  

In [6]:
def predict_arx(model, target_date, h):
    X = np.array([[
        log_p.loc[target_date - timedelta(days=1), h],
        log_p.loc[target_date - timedelta(days=2), h],
        log_p.loc[target_date - timedelta(days=7), h],
        log_p.loc[target_date - timedelta(days=1)].min(),
        log_load.loc[target_date, h],
        *[1.0 if target_date.dayofweek == k else 0.0 for k in range(6)]
    ]])
    return model.predict(X)[0]